In [20]:
# Import message classes for direct chat-style prompts.
from langchain_core.messages import SystemMessage, HumanMessage

In [21]:
# Load environment variables and configure the Groq chat model.
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)


In [22]:
# Build a simple message list and send it directly to the model.
question = HumanMessage('tell me about the earth in 3 points')
system = SystemMessage('You are elemetary teacher. You answer in short sentences.')

messages = [system, question]
response = llm.invoke(messages)

print(response.content)

1. Earth is the third planet from the Sun and the only known planet with life.  
2. It has diverse ecosystems, including oceans, forests, and mountains.  
3. Earth's surface is made of land and water, and it has a protective atmosphere.


## Prompt Template Classes

| Prompt Template Class | Description |
| --- | --- |
| `SystemMessagePromptTemplate` | Template for generating system messages that provide model context or instructions. |
| `HumanMessagePromptTemplate` | Template for generating user messages that represent user input or questions. |
| `AIMessagePromptTemplate` | Template for generating AI messages that represent assistant responses. |
| `PromptTemplate` | Basic template class for creating prompts with static text and variable placeholders. |
| `ChatPromptTemplate` | Template for creating prompts with a sequence of message types in chat format. |

In [ ]:
# Import prompt template classes for reusable prompt construction.
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        PromptTemplate,
                                        ChatPromptTemplate
                                        )

In [ ]:
# Define reusable templates with placeholders for dynamic values.
system = SystemMessagePromptTemplate.from_template(f"You are {{level}} level in the subject of {{subject}}. You answer in short sentences.")
human = HumanMessagePromptTemplate.from_template(f"tell me about the {{subject}} in {{points}} points")

## Passing Data to Prompt Templates

There are two common ways to pass data to prompt templates:

1. Format each prompt template individually.
2. Pass all variables directly to the combined chat prompt template.

In [ ]:
# Format each prompt template separately before combining them.
system.format(level="elementary", subject="cricket")
human.format(subject="cricket", points=3)

messages = [system, human]
messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level', 'subject'], input_types={}, partial_variables={}, template='You are {level} level in the subject of {subject}. You answer in short sentences.'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['points', 'subject'], input_types={}, partial_variables={}, template='tell me about the {subject} in {points} points'), additional_kwargs={})]

In [ ]:
# Combine the templates and pass all variables in a single step.
template = ChatPromptTemplate.from_messages([system, human])
template_data = template.invoke({"level": "elementary", "subject": "cricket", "points": 3})
template_data

ChatPromptValue(messages=[SystemMessage(content='You are elementary level in the subject of cricket. You answer in short sentences.', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me about the cricket in 3 points', additional_kwargs={}, response_metadata={})])

In [ ]:
# Send the fully formatted chat prompt to the model.
llm.invoke(template_data)

AIMessage(content='1. Cricket is a bat-and-ball game played between two teams of 11 players.  \n2. The goal is to score runs by hitting the ball and running between wickets.  \n3. It has formats like Test (long), ODI (medium), and T20 (short) matches.', additional_kwargs={'reasoning_content': "Okay, the user wants to know about cricket in three points. Let me start by recalling the basics. First, cricket is a bat-and-ball game played between two teams. That's a fundamental point. Next, the objective is to score runs by hitting the ball and running between wickets. That's the second point. For the third point, I should mention the different formats like Test, ODI, and T20. Wait, maybe I should explain each briefly. Also, the user is at an elementary level, so keep it simple. Let me check if I have three clear, concise points. Yes, that should cover the main aspects without getting too technical. Make sure each point is a short sentence as instructed.\n"}, response_metadata={'token_usage